## 1. Import Libraries and Setup


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder

## 2. Data Loading


In [2]:
# Configuration
MERGED_DATA_DIR = '../../data/merged'
FINAL_DATA_DIR = '../../data/final'

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

print("=" * 80)
print("Merged Dataset Preprocessing Pipeline")
print("=" * 80)
print(f"\nMerged data location: {MERGED_DATA_DIR}")
print(f"Final location: {FINAL_DATA_DIR}\n")

Merged Dataset Preprocessing Pipeline

Merged data location: ../../data/merged
Final location: ../../data/final



In [3]:
MERGED_FILE = os.path.join(MERGED_DATA_DIR, 'merged_1.csv')
FINAL_FILE = os.path.join(FINAL_DATA_DIR, 'encoded_unsupervised.csv')

In [4]:
# Load merged dataset
df = pd.read_csv(MERGED_FILE)

# Display initial info
print("=" * 80)
print("DATAFRAME BEFORE ENCODING")
print("=" * 80)

print(f"\nShape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

DATAFRAME BEFORE ENCODING

Shape: (32548, 21)

Columns (21): ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'avg_clicks_per_day', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay', 'sites_revisit_ratio', 'activity_diversity_ratio']

Data types:
code_module                       str
code_presentation                 str
id_student                      int64
gender                            str
region                            str
highest_education                 str
imd_band                          str
age_band                          str
num_of_prev_attempts            int64
studied_credits                 int64
disability                        str
final_result                      str
date_registration             float64
avg_clicks_per_day    

## 3. Clean and Prepare Data

Drop id_student and region to remove PII and irrelevant features.


In [5]:
# Drop columns: PII and irrelevant features
columns_to_drop = ['id_student', 'region']
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df_encoded = df.drop(columns=columns_to_drop)

print("\n" + "=" * 80)
print("STEP 1: DATA CLEANING")
print("=" * 80)
print(f"\nColumns dropped (PII + irrelevant): {columns_to_drop}")
print(f"Shape after dropping: {df_encoded.shape}")
print(f"Remaining columns: {df_encoded.columns.tolist()}")


STEP 1: DATA CLEANING

Columns dropped (PII + irrelevant): ['id_student', 'region']
Shape after dropping: (32548, 19)
Remaining columns: ['code_module', 'code_presentation', 'gender', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'avg_clicks_per_day', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay', 'sites_revisit_ratio', 'activity_diversity_ratio']


## 4. Apply Ordinal Mappings

Map educational levels, age bands, and socioeconomic indices to ordinal values.


In [6]:
print("\n" + "=" * 80)
print("STEP 2: ORDINAL MAPPINGS")
print("=" * 80)

# 1. Highest Education Mapping
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

if 'highest_education' in df_encoded.columns:
    print(f"\nBefore education mapping:")
    print(df_encoded['highest_education'].value_counts())
    
    df_encoded['highest_education'] = df_encoded['highest_education'].map(education_mapping)
    print(f"After education mapping:")
    print(df_encoded['highest_education'].value_counts(dropna=False))



STEP 2: ORDINAL MAPPINGS

Before education mapping:
highest_education
A Level or Equivalent          14026
Lower Than A Level             13138
HE Qualification                4725
No Formal quals                  346
Post Graduate Qualification      313
Name: count, dtype: int64
After education mapping:
highest_education
2    14026
1    13138
3     4725
0      346
4      313
Name: count, dtype: int64


In [7]:
# 2. IMD Band Mapping
imd_mapping = {
    '0-10%': 0,
    '10-20%': 1,
    '20-30%': 2,
    '30-40%': 3,
    '40-50%': 4,
    '50-60%': 5,
    '60-70%': 6,
    '70-80%': 7,
    '80-90%': 8,
    '90-100%': 9
}

if 'imd_band' in df_encoded.columns:
    print(f"\nBefore imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))
    
    # Apply mapping
    df_encoded['imd_band'] = df_encoded['imd_band'].map(imd_mapping)
    print(f"After imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))


Before imd_band mapping:
imd_band
10-20%     4242
20-30%     3651
0-10%      3618
30-40%     3541
40-50%     3250
50-60%     3130
60-70%     2899
70-80%     2877
80-90%     2758
90-100%    2582
Name: count, dtype: int64
After imd_band mapping:
imd_band
1    4242
2    3651
0    3618
3    3541
4    3250
5    3130
6    2899
7    2877
8    2758
9    2582
Name: count, dtype: int64


In [8]:
# 3. Final Result Mapping (Target Variable - Ordinal)
final_result_mapping = {
    'Distinction': 3,
    'Pass': 2,
    'Fail': 1,
    'Withdrawn': 0
}

if 'final_result' in df_encoded.columns:
    print(f"\nBefore final_result mapping:")
    print(df_encoded['final_result'].value_counts())
    
    df_encoded['final_result'] = df_encoded['final_result'].map(final_result_mapping)
    print(f"After final_result mapping:")
    print(df_encoded['final_result'].value_counts(dropna=False))
    print(f"Mapping: Distinction=3 (Highest), Pass=2, Fail=1, Withdrawn=0 (Lowest)")



Before final_result mapping:
final_result
Pass           12360
Withdrawn      10117
Fail            7047
Distinction     3024
Name: count, dtype: int64
After final_result mapping:
final_result
2    12360
0    10117
1     7047
3     3024
Name: count, dtype: int64
Mapping: Distinction=3 (Highest), Pass=2, Fail=1, Withdrawn=0 (Lowest)


## 5. Perform One-Hot Encoding

Use pd.get_dummies for categorical features: code_module, code_presentation, and gender. Drop first category to avoid multicollinearity.


In [9]:
print("\n" + "=" * 80)
print("STEP 3: ONE-HOT ENCODING")
print("=" * 80)

# Identify categorical columns for one-hot encoding
categorical_cols = ['code_module']
categorical_cols = [col for col in categorical_cols if col in df_encoded.columns]

print(f"\nColumns to one-hot encode: {categorical_cols}")

# Apply one-hot encoding using OneHotEncoder (drop first category to avoid multicollinearity)
encoder = OneHotEncoder(
    drop='first',
    sparse_output=False,
    dtype=int
)

# Transform only the categorical columns
encoded_array = encoder.fit_transform(df_encoded[categorical_cols])

# Get feature names for the encoded columns
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

# Create a dataframe from the encoded array
df_encoded_cats = pd.DataFrame(encoded_array, columns=encoded_feature_names, index=df_encoded.index)

# Drop original categorical columns and concatenate with encoded columns
df_encoded = pd.concat(
    [df_encoded.drop(columns=categorical_cols), df_encoded_cats],
    axis=1
)


STEP 3: ONE-HOT ENCODING

Columns to one-hot encode: ['code_module']


In [10]:

print(f"\nShape after one-hot encoding: {df_encoded.shape}")
print(f"\nNew columns created:")
new_cols = [col for col in df_encoded.columns if any(cat in col for cat in categorical_cols)]
print(new_cols)

print(f"\nTotal columns now: {len(df_encoded.columns)}")


Shape after one-hot encoding: (32548, 24)

New columns created:
['code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF', 'code_module_GGG']

Total columns now: 24


## 6. Scale Numerical Features

Scaling strategy based on feature characteristics:

- **RobustScaler (High Outliers)**: avg_clicks_per_day, studied_credits, sites_revisit_ratio, activity_diversity_ratio, avg_submission_delay, total_submissions
- **MinMaxScaler (Bounded)**: tma_cma_weighted_score, highest_education, imd_band, final_result
- **Log Transform + StandardScaler (Skewed/Count)**: late_submissions_count, num_of_prev_attempts
- **StandardScaler (Normal)**: date_registration, module_presentation_length


In [11]:
print("\n" + "=" * 80)
print("STEP 4: NUMERICAL FEATURE SCALING")
print("=" * 80)

# Define feature groups with different scaling strategies
# Based on feature characteristics guide
features_robust_outliers = ['avg_clicks_per_day', 'studied_credits', 'sites_revisit_ratio', 
                            'activity_diversity_ratio', 'avg_submission_delay', 'total_submissions']
features_minmax_bounded = ['tma_cma_weighted_score', 'highest_education', 'imd_band', 'final_result']
features_log_standard = ['late_submissions_count', 'num_of_prev_attempts']
features_standard_normal = ['date_registration', 'module_presentation_length']

# Check which features exist in the dataframe
robust_present = [col for col in features_robust_outliers if col in df_encoded.columns]
minmax_present = [col for col in features_minmax_bounded if col in df_encoded.columns]
log_standard_present = [col for col in features_log_standard if col in df_encoded.columns]
standard_present = [col for col in features_standard_normal if col in df_encoded.columns]

print(f"\n--- Feature Scaling Strategy ---")
print(f"RobustScaler (High Outliers): {robust_present}")
print(f"MinMaxScaler (Bounded): {minmax_present}")
print(f"Log + StandardScaler (Skewed/Count): {log_standard_present}")
print(f"StandardScaler (Normal): {standard_present}")

# 1. Apply RobustScaler to high outlier features
if robust_present:
    print(f"\n--- RobustScaler (High Outliers) ---")
    print(f"Before scaling:")
    print(df_encoded[robust_present].describe())
    
    scaler_robust = RobustScaler()
    df_encoded[robust_present] = scaler_robust.fit_transform(df_encoded[robust_present])
    
    print(f"After scaling:")
    print(df_encoded[robust_present].describe())

# 2. Apply MinMaxScaler to bounded features (including ordinal)
if minmax_present:
    print(f"\n--- MinMaxScaler (Bounded) ---")
    print(f"Before scaling:")
    print(df_encoded[minmax_present].describe())
    
    scaler_minmax = MinMaxScaler()
    df_encoded[minmax_present] = scaler_minmax.fit_transform(df_encoded[minmax_present])
    
    print(f"After scaling (should be in [0, 1]):")
    print(df_encoded[minmax_present].describe())

# 3. Apply Log Transform + StandardScaler to skewed count features
if log_standard_present:
    print(f"\n--- Log Transform + StandardScaler (Skewed/Count) ---")
    print(f"Before scaling:")
    print(df_encoded[log_standard_present].describe())
    
    # Apply log1p transform first (handles zeros)
    df_encoded[log_standard_present] = np.log1p(df_encoded[log_standard_present])
    print(f"After log1p transform:")
    print(df_encoded[log_standard_present].describe())
    
    # Then apply StandardScaler
    scaler_standard_log = StandardScaler()
    df_encoded[log_standard_present] = scaler_standard_log.fit_transform(df_encoded[log_standard_present])
    
    print(f"After StandardScaler:")
    print(df_encoded[log_standard_present].describe())

# 4. Apply StandardScaler to normal features
if standard_present:
    print(f"\n--- StandardScaler (Normal) ---")
    print(f"Before scaling:")
    print(df_encoded[standard_present].describe())
    
    scaler_standard = StandardScaler()
    df_encoded[standard_present] = scaler_standard.fit_transform(df_encoded[standard_present])
    
    print(f"After scaling:")
    print(df_encoded[standard_present].describe())

if not (robust_present or minmax_present or log_standard_present or standard_present):
    print("\nWarning: No features found to scale")


STEP 4: NUMERICAL FEATURE SCALING

--- Feature Scaling Strategy ---
RobustScaler (High Outliers): ['avg_clicks_per_day', 'studied_credits', 'sites_revisit_ratio', 'activity_diversity_ratio', 'avg_submission_delay', 'total_submissions']
MinMaxScaler (Bounded): ['tma_cma_weighted_score', 'highest_education', 'imd_band', 'final_result']
Log + StandardScaler (Skewed/Count): ['late_submissions_count', 'num_of_prev_attempts']
StandardScaler (Normal): ['date_registration', 'module_presentation_length']

--- RobustScaler (High Outliers) ---
Before scaling:


       avg_clicks_per_day  studied_credits  sites_revisit_ratio  \
count        32548.000000     32548.000000         32548.000000   
mean             3.744569        79.714422             3.388834   
std              2.120344        41.046179             2.276461   
min              0.000000        30.000000             0.000000   
25%              2.534611        60.000000             2.017857   
50%              3.690916        60.000000             3.159391   
75%              5.020506       120.000000             4.407744   
max             33.211540       655.000000            22.461538   

       activity_diversity_ratio  avg_submission_delay  total_submissions  
count              32548.000000          32548.000000       32548.000000  
mean                   0.201415             -9.703851           5.343032  
std                    0.161790             23.872070           4.324641  
min                    0.000000           -236.000000           0.000000  
25%                  

In [12]:
df_encoded.head()

,code_presentation,gender,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,...,late_submissions_count,avg_submission_delay,sites_revisit_ratio,activity_diversity_ratio,code_module_BBB,code_module_CCC,code_module_DDD,code_module_EEE,code_module_FFF,code_module_GGG
0,2013J,M,0.75,1.000000,55+,-0.368196,3.0,N,0.666667,-1.818699,...,-0.828366,-0.380537,-0.051478,0.370037,0,0,0,0,0,0
1,2013J,F,0.75,0.222222,35-55,-0.368196,0.0,N,0.666667,0.333158,...,0.651325,0.000000,0.456343,0.925370,0,0,0,0,0,0
2,2013J,F,0.50,0.333333,35-55,-0.368196,0.0,Y,0.000000,-0.458563,...,-0.828366,0.000000,-0.123753,-0.261894,0,0,0,0,0,0
3,2013J,F,0.50,0.555556,35-55,-0.368196,0.0,N,0.666667,0.353459,...,-0.828366,-0.422819,1.464147,0.887071,0,0,0,0,0,0
4,2013J,F,0.25,0.555556,0-35,-0.368196,0.0,N,0.666667,-2.163809,...,1.584907,2.410067,0.649709,0.580680,0,0,0,0,0,0


## 7. Downcasting Numeric Types for Memory Optimization

Precision Considerations

- float64: ~15 decimal digits precision
- float32: ~7 decimal digits precision
- For scaled features\*\*: Loss is negligible since StandardScaler/MinMaxScaler normalize values to [-1, 1] or [0, 1] ranges
- ML Reality: Most algorithms cannot leverage the extra precision of float64 anyway


In [13]:
print("\n" + "=" * 80)
print("STEP 5: DOWNCAST DTYPES FOR MEMORY OPTIMIZATION")
print("=" * 80)

# Calculate memory before downcasting
memory_before = df_encoded.memory_usage(deep=True).sum() / 1024**2  # Convert to MB
print(f"\nMemory before downcasting: {memory_before:.2f} MB")

# Downcast float64 to float32
float_cols = df_encoded.select_dtypes(include=['float64']).columns
if len(float_cols) > 0:
    print(f"\nDowncasting float64 → float32: {list(float_cols)}")
    df_encoded[float_cols] = df_encoded[float_cols].astype('float32')

# Downcast int64 to int32 (or int16 if values are small)
int_cols = df_encoded.select_dtypes(include=['int64']).columns
if len(int_cols) > 0:
    print(f"\nDowncasting int64 columns: {list(int_cols)}")
    for col in int_cols:
        # Check if values fit in int32
        if df_encoded[col].max() < 2**31 - 1 and df_encoded[col].min() > -2**31:
            df_encoded[col] = df_encoded[col].astype('int32')
            print(f"  {col}: int64 → int32")
        else:
            print(f"  {col}: keeping int64 (values exceed int32 range)")

# Calculate memory after downcasting
memory_after = df_encoded.memory_usage(deep=True).sum() / 1024**2  # Convert to MB
memory_saved = memory_before - memory_after
percentage_saved = (memory_saved / memory_before) * 100

print(f"\nMemory after downcasting: {memory_after:.2f} MB")
print(f"Memory saved: {memory_saved:.2f} MB ({percentage_saved:.1f}%)")

print(f"\nData types after downcasting:")
print(df_encoded.dtypes)


STEP 5: DOWNCAST DTYPES FOR MEMORY OPTIMIZATION

Memory before downcasting: 11.40 MB

Downcasting float64 → float32: ['highest_education', 'imd_band', 'num_of_prev_attempts', 'studied_credits', 'final_result', 'date_registration', 'avg_clicks_per_day', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay', 'sites_revisit_ratio', 'activity_diversity_ratio']

Downcasting int64 columns: ['code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF', 'code_module_GGG']
  code_module_BBB: int64 → int32
  code_module_CCC: int64 → int32
  code_module_DDD: int64 → int32
  code_module_EEE: int64 → int32
  code_module_FFF: int64 → int32
  code_module_GGG: int64 → int32

Memory after downcasting: 8.92 MB
Memory saved: 2.48 MB (21.8%)

Data types after downcasting:
code_presentation                 str
gender                            str
highest_education             float32
imd_band         

## 8. Export Final Encoded Dataset

Save the final encoded and scaled dataframe to data/final/encoded_unsupervised.csv.


In [14]:

print("\n" + "=" * 80)
print("STEP 6: EXPORT FINAL ENCODED DATASET")
print("=" * 80)

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

# Save to data/final/
FINAL_FILE_PATH = os.path.join(FINAL_DATA_DIR, 'encoded_supervised_major.csv')
df_encoded.to_csv(FINAL_FILE_PATH, index=False)

print(f"\n✓ Final encoded dataset saved to: {FINAL_FILE_PATH}")

print(f"\nFinal dataframe information:")
print(f"  Shape: {df_encoded.shape}")
print(f"  Columns: {len(df_encoded.columns)}")
print(f"  Data types:\n{df_encoded.dtypes}")
print(f"\nMissing values:")
print(df_encoded.isnull().sum())

print(f"\nFirst few rows:")
print(df_encoded.head())

print("\n" + "=" * 80)
print("ENCODING PIPELINE COMPLETE")
print("=" * 80)


STEP 6: EXPORT FINAL ENCODED DATASET

✓ Final encoded dataset saved to: ../../data/final\encoded_supervised_major.csv

Final dataframe information:
  Shape: (32548, 24)
  Columns: 24
  Data types:
code_presentation                 str
gender                            str
highest_education             float32
imd_band                      float32
age_band                          str
num_of_prev_attempts          float32
studied_credits               float32
disability                        str
final_result                  float32
date_registration             float32
avg_clicks_per_day            float32
module_presentation_length    float32
tma_cma_weighted_score        float32
total_submissions             float32
late_submissions_count        float32
avg_submission_delay          float32
sites_revisit_ratio           float32
activity_diversity_ratio      float32
code_module_BBB                 int32
code_module_CCC                 int32
code_module_DDD                 int32
code